# Fehleranalyse: Multi-Task-MLP



**Autor:** duac1011 · **Datum:** 2026-10-13



**Ziel:** Erklären, warum das Multi-Task-MLP mit PCA-20 unter der kombinierten RBF-SVM liegt und welche Fehlerbilder die Differenz bestimmen (#32).

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from awp2.config import (
    AEZ_COL,
    MODEL_DOCS_FIGURES_DIR,
    MONTH_COL,
    RECALL_COL,
    SEED,
    SUPPORT_COL,
)
from awp2.data import band_columns, combined_label, load_split
from awp2.evaluation import compare_predictions, error_breakdown, recall_by_class
from awp2.experiment import RunConfig, out_of_fold_predictions, run
from awp2.models import MultiTaskMLPClassifier, make_combined_svm
from awp2.plots import (
    plot_error_breakdown,
    plot_error_rate_by,
    plot_recall_comparison,
    plot_spectra,
    save_doc_figure,
)
from awp2.preprocessing import PreprocessingConfig, ndvi

split = load_split()
pca_preprocessing = PreprocessingConfig(
    use_spectral_standard_scale=True,
    spectral_pca_components=20,
)
approaches = {
    "Multi-Task-MLP (PCA-20)": (
        "multi-task",
        MultiTaskMLPClassifier(
            hidden_units=64,
            hidden_layers=2,
            dropout=0.2,
            weight_decay=0.0,
            random_state=SEED,
        ),
        pca_preprocessing,
    ),
    "Kombinierte SVM": (
        "combined",
        make_combined_svm().set_params(estimator__C=100.0, estimator__gamma=0.003),
        PreprocessingConfig(scale=True, use_cyclic_month=False),
    ),
}
results = {
    label: run(
        model,
        RunConfig(
            name="multi_task_error_analysis_refit",
            approach=approach,
            preprocessing=preprocessing,
            track=False,
            log_model=False,
        ),
    )
    for label, (approach, model, preprocessing) in approaches.items()
}
predictions = {label: result.y_pred for label, result in results.items()}
pd.DataFrame({label: result.metrics.model_dump() for label, result in results.items()}).round(3)

## 2. Fehlerzerlegung



Eine falsche Kultur macht das kombinierte Paar zwangsläufig falsch. Eine richtige Kultur mit falschem Stadium isoliert dagegen die Fehler, die im Stadiumskopf oder bei der Paarentscheidung entstehen.

In [ ]:
breakdowns = {

    label: error_breakdown(split.y_val, prediction)

    for label, prediction in predictions.items()

}

breakdown_ax = plot_error_breakdown(breakdowns)

breakdown_ax.set_title("Fehlerzerlegung auf der Validierung")

save_doc_figure(

    breakdown_ax.figure,

    "multi_task_error_breakdown",

    MODEL_DOCS_FIGURES_DIR,

)

pd.DataFrame({label: breakdown.model_dump() for label, breakdown in breakdowns.items()}).round(3)

## 3. Schwache Crop/Stage-Paare



Die kombinierte Balanced Accuracy ist der Mittelwert der Recalls über die wahren Crop/Stage-Paare. Deshalb senken auch seltene Paare mit wenigen Zeilen den Wert deutlich; Support und Recall müssen gemeinsam gelesen werden.

In [ ]:
recalls = {

    label: recall_by_class(split.y_val, prediction)

    for label, prediction in predictions.items()

}

recall_figure = plot_recall_comparison(recalls, top=12)

save_doc_figure(

    recall_figure,

    "multi_task_error_recall",

    MODEL_DOCS_FIGURES_DIR,

)

recall_table = pd.DataFrame({label: recall[RECALL_COL] for label, recall in recalls.items()})

recall_table[SUPPORT_COL] = recalls["Kombinierte SVM"][SUPPORT_COL]

recall_table["Multi-Task minus SVM"] = (

    recall_table["Multi-Task-MLP (PCA-20)"] - recall_table["Kombinierte SVM"]

)

recall_table.sort_values("Multi-Task minus SVM").round(3)

## 4. Unterschiede zur kombinierten SVM



Der exakte McNemar-Test betrachtet je Zeile nur, ob genau ein Modell das gesamte Paar trifft. Er testet damit die Stichprobenkorrektheit, nicht die Balanced Accuracy; die häufigen Klassen dominieren den Test.

In [ ]:
comparison = compare_predictions(

    split.y_val,

    predictions["Multi-Task-MLP (PCA-20)"],

    predictions["Kombinierte SVM"],

)

mlp_pair = combined_label(predictions["Multi-Task-MLP (PCA-20)"])

svm_pair = combined_label(predictions["Kombinierte SVM"])

true_pair = combined_label(split.y_val)

confusions = pd.DataFrame(

    {

        "wahr": true_pair,

        "Multi-Task-MLP": mlp_pair,

        "Kombinierte SVM": svm_pair,

    }

)

top_mlp_confusions = (

    confusions.loc[confusions["wahr"] != confusions["Multi-Task-MLP"], ["wahr", "Multi-Task-MLP"]]

    .value_counts()

    .head(10)

    .rename("Zeilen")

)

comparison.model_dump(), top_mlp_confusions

## 5. Zeitliche und räumliche Fehlerprofile



Paarfehler nach Monat und AEZ zeigen, ob die Lücke an bestimmten Aufnahmebedingungen liegt. Gruppen mit kleiner Fallzahl liefern nur Hinweise, keine belastbare Ursache.

In [ ]:
pair_errors = {
    label: combined_label(split.y_val) != combined_label(prediction)
    for label, prediction in predictions.items()
}

month_ax = plot_error_rate_by(pair_errors, split.X_val[MONTH_COL], xlabel="Monat")
month_ax.set_title("Paarfehler nach Monat")
save_doc_figure(month_ax.figure, "multi_task_error_month", MODEL_DOCS_FIGURES_DIR)

aez_ax = plot_error_rate_by(pair_errors, split.X_val[AEZ_COL], xlabel="AEZ")
aez_ax.set_title("Paarfehler nach AEZ")
save_doc_figure(aez_ax.figure, "multi_task_error_aez", MODEL_DOCS_FIGURES_DIR)

(
    pd.DataFrame(
        {
            label: error.groupby(split.X_val[MONTH_COL]).mean()
            for label, error in pair_errors.items()
        }
    ).round(3),
    pd.DataFrame(
        {
            label: error.groupby(split.X_val[AEZ_COL]).mean()
            for label, error in pair_errors.items()
        }
    ).round(3),
)

## 6. Gegenprobe mit Out-of-Fold-Vorhersagen



Die gleichen festen MLP-Einstellungen werden in jedem Trainingsfold neu gefittet. Stimmen die Monatsmuster mit der Validierung überein, sind sie eine robustere Grundlage für nächste Hypothesen als ein einzelner Holdout.

In [ ]:
oof_predictions = out_of_fold_predictions(
    approaches["Multi-Task-MLP (PCA-20)"][1],
    RunConfig(
        name="multi_task_error_analysis_oof",
        approach="multi-task",
        preprocessing=pca_preprocessing,
        track=False,
    ),
)
oof_pair_errors = combined_label(split.y_train) != combined_label(oof_predictions)
pd.DataFrame(
    {
        "Validierung": (
            pair_errors["Multi-Task-MLP (PCA-20)"]
            .groupby(split.X_val[MONTH_COL])
            .mean()
        ),
        "Training (out-of-fold)": oof_pair_errors.groupby(split.X_train[MONTH_COL]).mean(),
        "Zeilen Validierung": split.X_val[MONTH_COL].value_counts(),
        "Zeilen OOF": split.X_train[MONTH_COL].value_counts(),
    }
).round(3)

## 7. Stärkste Verwechslung spektral einordnen



Die häufigste Verwechslung wird den korrekt erkannten Zeilen desselben wahren Paars und den echten Zeilen des vorhergesagten Paars gegenübergestellt. Die Kurven zeigen Mittelwerte und Streuung; sie belegen Ähnlichkeit, aber keine kausale Ursache.

In [ ]:
top_true, top_predicted = top_mlp_confusions.index[0]

top_confusion = (true_pair == top_true) & (mlp_pair == top_predicted)

groups = {

    f"{top_true}, verwechselt als {top_predicted}": top_confusion,

    f"{top_true}, richtig erkannt": (true_pair == top_true) & (mlp_pair == top_true),

    f"{top_predicted}, echte Referenz": true_pair == top_predicted,

}

top_frame = pd.concat(

    [split.X_val.loc[mask].assign(Gruppe=label) for label, mask in groups.items()]

)

top_ax = plot_spectra(top_frame, by="Gruppe", title=f"{top_true} -> {top_predicted}")

save_doc_figure(

    top_ax.figure,

    "multi_task_error_top_confusion_spectra",

    MODEL_DOCS_FIGURES_DIR,

)

bands = band_columns(top_frame)

pd.DataFrame(

    {

        "Zeilen": top_frame.groupby("Gruppe").size(),

        "NDVI Median": top_frame.groupby("Gruppe").apply(lambda group: ndvi(group).median()),

        "mittlere Reflektanz": top_frame.groupby("Gruppe")[bands].mean().mean(axis=1),

        "Monat": top_frame.groupby("Gruppe")[MONTH_COL].agg(lambda values: values.mode().iat[0]),

        "AEZ": top_frame.groupby("Gruppe")[AEZ_COL].agg(lambda values: values.mode().iat[0]),

    }

).round(2)

## Fazit



- Der Abstand zur kombinierten SVM ist reproduzierbar: BAcc kombiniert 0,764 statt 0,866. Beide Modelle erzeugen ausschließlich gültige Paare; die Paarbeschränkung ist daher nicht die Ursache.

- Das MLP verliert gleichzeitig bei Kultur (10,6 % statt 8,4 % falsche Kulturen) und bei richtig erkannter Kultur im Stadium (6,7 % statt 3,9 %). Die SVM trifft 139 vollständige Paare exklusiv, das MLP nur 55 ($p = 1{,}4\cdot10^{-9}$).

- Besonders schwach sind Übergangs- und Seneszenzpaare, etwa `winter_wheat|Critical` (0,000 statt 0,882 Recall) und `cotton|Emerge_VEarly` (0,615 statt 0,769). Das Monatsprofil bestätigt mit Out-of-Fold-Vorhersagen hohe Fehlerraten im Mai, Juni und Oktober.

- Die häufigste einzelne Verwechslung, `cotton|Emerge_VEarly → cotton|Early_Mid`, passt zum Übergangscharakter: Die verwechslten Juni-Spektren liegen in NDVI und Helligkeit näher an der späteren Referenz als an korrekt erkannten frühen Baumwollzeilen.

- Die kombinierte SVM lernt die 23 Crop/Stage-Paare direkt. Das MLP optimiert dagegen zwei getrennte, gleich gewichtete Aufgaben und verbindet sie erst bei der Dekodierung. Ob die gemeinsame Repräsentation gegenüber zwei getrennten MLPs überhaupt hilft, bleibt offen; ein kontrollierter CV-Vergleich ist der nächste Schritt.